# SEC MD&A Pipeline
### Table Extraction → Text Chunking → Embeddings → Vector Store

**Steps**
1. Load SEC MD&A data from Parquet  
2. Extract tables (HTML + whitespace-aligned) → SQLite  
3. Chunk cleaned text — MD&A section-aware with 15 % overlap; **fixed-size fallback** when no sections are detected  
4. Generate sentence-transformer embeddings  
5. Store embeddings in **Redis Vector Search** (HNSW / COSINE) — **falls back automatically to SQLite** if Redis Stack is unreachable  
6. Run KNN and filtered search demos against whichever backend is active  

**Prerequisites**
```bash
pip install sentence-transformers redis pandas pyarrow numpy
# Redis Stack (optional — SQLite is used automatically if unavailable)
docker run -d -p 6379:6379 redis/redis-stack-server:latest
```

In [ ]:
# ── 0. Install dependencies (run once) ────────────────────────────────────
import subprocess, sys

PACKAGES = [
    "sentence-transformers",
    "redis",
    "pandas",
    "pyarrow",
    "numpy",
]
for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages ready.")

In [ ]:
# ── 1. Configuration ───────────────────────────────────────────────────────
import os, sys, json, re, struct, sqlite3
from pathlib import Path
from typing  import List, Dict, Optional

import numpy as np
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
# Directory containing table_extraction.py, text_chunking.py, embeddings.py
PIPELINE_DIR = Path("/home/d1ttb/mda_wip/mda_pipeline")

# Parquet with columns: accession, year, mda_text, cik, name, sic,
#                       ein, address, naics3, naics17Title, is_ambiguous_naics3
PARQUET_PATH = Path("/home/d1ttb/mda_wip/mda_merged_sic_naics_24_25.parquet")

# SQLite — tables store AND embedding fallback store share this file
SQLITE_PATH  = "sec_tables.db"

# ── Redis ──────────────────────────────────────────────────────────────────
REDIS_HOST   = "localhost"
REDIS_PORT   = 6379
REDIS_DB     = 0
REDIS_INDEX  = "sec_mda_chunks"
REDIS_PREFIX = "sec:chunk:"

# ── Chunking ───────────────────────────────────────────────────────────────
CHUNK_SIZE_TOKENS = 128     # ~512 chars per chunk
OVERLAP_PERCENT   = 15
CHUNKING_STRATEGY = "mda"   # "mda" → section-aware + fixed-size fallback
                             # "size" → always fixed-size

# ── Embedding ──────────────────────────────────────────────────────────────
EMBED_MODEL = "multi-qa-MiniLM-L6-cos-v1"   # dim = 384
EMBED_BATCH = 64

# Max documents to process (set to None for full dataset)
MAX_DOCS = 50

# ── Pipeline output ────────────────────────────────────────────────────────
OUTPUT_JSON = "embedded_chunks_export.json"

sys.path.insert(0, str(PIPELINE_DIR))

print(f"Pipeline dir  : {PIPELINE_DIR}")
print(f"Parquet path  : {PARQUET_PATH}")
print(f"Redis target  : {REDIS_HOST}:{REDIS_PORT}  index={REDIS_INDEX}")
print(f"SQLite path   : {SQLITE_PATH}  (tables + embedding fallback)")
print(f"Chunk strategy: {CHUNKING_STRATEGY}  size={CHUNK_SIZE_TOKENS} tokens  overlap={OVERLAP_PERCENT}%")
print(f"Embed model   : {EMBED_MODEL}")

In [ ]:
# ── 2. Import pipeline modules ─────────────────────────────────────────────
from table_extraction import TableExtractor, ExtractedTable, SQLiteTableStore
from text_chunking    import TextChunker, TextChunk
from embeddings       import EmbeddingGenerator, EmbeddedChunk


def divider(title: str) -> None:
    print(f"\n{'='*70}\n  {title}\n{'='*70}")


# Lightweight document wrapper (avoids requiring the full SECDocument dataclass)
class MDAdoc:
    __slots__ = [
        "document_id", "accession", "year", "cik", "name",
        "sic", "ein", "address", "naics3", "naics17Title",
        "is_ambiguous_naics3", "mda_text", "metadata",
    ]
    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)


print("Pipeline modules imported.")

In [ ]:
# ── 3. Load data ───────────────────────────────────────────────────────────
if PARQUET_PATH.exists():
    df_mda = pd.read_parquet(PARQUET_PATH)
    print(f"Loaded {len(df_mda):,} rows  |  columns: {list(df_mda.columns)}")

    subset = df_mda.iloc[:MAX_DOCS] if MAX_DOCS else df_mda

    documents = [
        MDAdoc(
            document_id         = f"{row['cik']}_{row['year']}_{row['accession']}",
            accession           = row.get("accession", ""),
            year                = str(row.get("year", "")),
            cik                 = str(row.get("cik", "")),
            name                = row.get("name", ""),
            sic                 = str(row.get("sic", "")),
            ein                 = str(row.get("ein", "")),
            address             = row.get("address", ""),
            naics3              = str(row.get("naics3", "")),
            naics17Title        = row.get("naics17Title", ""),
            is_ambiguous_naics3 = bool(row.get("is_ambiguous_naics3", False)),
            mda_text            = row.get("mda_text", ""),
            metadata            = {},
        )
        for _, row in subset.iterrows()
    ]
    print(f"Prepared {len(documents):,} documents")

else:
    # ── Synthetic single-document fallback for quick testing ───────────────
    print(f"[warn] Parquet not found — using synthetic demo document")

    DEMO_TEXT = """
ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS

Overview

The company delivered strong results in fiscal 2024 with revenue growth of 12%
and expanded operating margins. Our diversified portfolio of utility assets
continued to generate stable cash flows.

Results of Operations

Revenue increased to $4.0 billion, up from $3.6 billion in the prior year.
Operating income rose to $1.1 billion, driven by lower fuel costs and higher
transmission volumes.

<table>
<tr><th>Year</th><th>Revenue ($M)</th><th>Op. Income ($M)</th></tr>
<tr><td>2024</td><td>4,000</td><td>1,100</td></tr>
<tr><td>2023</td><td>3,600</td><td>  980</td></tr>
<tr><td>2022</td><td>3,200</td><td>  870</td></tr>
</table>

Liquidity and Capital Resources

Cash provided by operations totalled $1.3 billion. Capital expenditures were
$1.5 billion, funding infrastructure upgrades and renewable-energy additions.
We maintain $2.0 billion of revolving credit capacity.

Cash Flows

Operating cash flow improved $325 million year-over-year. Investing outflows
reflect our five-year $9.8 billion capital plan.

Risk Factors

Macroeconomic conditions and supply chain issues continue to represent key
risks. We mitigate through diversified sourcing and hedging programs.

Environmental Matters

We are committed to reducing carbon emissions 80% by 2030. Capital investments
in solar, wind, and battery storage support this trajectory.
"""

    documents = [
        MDAdoc(
            document_id="DEMO_0000001_2024", accession="0000001-24-000001",
            year="2024", cik="0000001", name="Demo Utility Corp",
            sic="4911", ein="12-3456789", address="123 Main St",
            naics3="221", naics17Title="Utilities",
            is_ambiguous_naics3=False, mda_text=DEMO_TEXT, metadata={},
        )
    ]
    print(f"Demo document created ({len(DEMO_TEXT):,} chars)")

In [ ]:
# ── 4. Table extraction ────────────────────────────────────────────────────
divider("STEP 2 — TABLE EXTRACTION")

extractor = TableExtractor(min_formatted_rows=3)
cleaned_docs, all_tables = extractor.extract_from_documents(documents)

print(f"  Documents in        : {len(documents):,}")
print(f"  Tables extracted    : {len(all_tables):,}")

# Preview first 3 tables
for i, t in enumerate(all_tables[:3]):
    print(f"\n  ── Table {i} ──")
    print(f"     ID    : {t.table_id}")
    print(f"     Title : {t.table_title}")
    print(f"     Rows  : {len(t.table_data)}")
    for row in t.table_data[:4]:
        print(f"       {row}")
    if len(t.table_data) > 4:
        print(f"       … ({len(t.table_data)-4} more rows)")

In [ ]:
# ── 5. Persist tables to SQLite ────────────────────────────────────────────
divider("STEP 2b — PERSIST TABLES → SQLITE")

with SQLiteTableStore(db_path=SQLITE_PATH) as store:
    n_stored = store.store_tables(all_tables)
    stats    = store.statistics()

print("\n  SQLite statistics:")
for k, v in stats.items():
    print(f"    {k}: {v}")

In [ ]:
# ── 6. Text chunking ──────────────────────────────────────────────────────
# Strategy "mda":  tries MD&A section detection first; automatically falls
#                  back to fixed-size chunks (chunk_by_size) when fewer than
#                  two section headers are found in a document.
# Strategy "size": always uses fixed-size chunks — useful as a baseline or
#                  when the text lacks standard MD&A structure.
# Both strategies produce chunks of CHUNK_SIZE_TOKENS tokens with
# OVERLAP_PERCENT % overlap and respect sentence / line boundaries.
# ──────────────────────────────────────────────────────────────────────────
divider(f"STEP 3 — TEXT CHUNKING  (strategy='{CHUNKING_STRATEGY}')")

chunker = TextChunker(
    chunk_size_tokens=CHUNK_SIZE_TOKENS,
    overlap_percent=OVERLAP_PERCENT,
)

# Instrument chunk_by_mda_structure to count fallbacks
_fallback_docs: List[str] = []
_orig_mda = chunker.chunk_by_mda_structure

def _instrumented_mda(text, source_id, metadata):
    headers = chunker._find_mda_sections(text)
    if not headers:
        _fallback_docs.append(source_id)
    return _orig_mda(text, source_id, metadata)

chunker.chunk_by_mda_structure = _instrumented_mda

chunks = chunker.run(cleaned_docs, strategy=CHUNKING_STRATEGY)

print(f"\n  Chunk size          : {chunker.chunk_size} chars (~{CHUNK_SIZE_TOKENS} tokens)")
print(f"  Overlap             : {chunker.overlap} chars ({OVERLAP_PERCENT}%)")
print(f"  Total chunks        : {len(chunks):,}")
print(f"  Fixed-size fallbacks: {len(_fallback_docs)} doc(s)")
if _fallback_docs:
    for doc_id in _fallback_docs[:10]:
        print(f"    - {doc_id}")
    if len(_fallback_docs) > 10:
        print(f"    … and {len(_fallback_docs)-10} more")

In [ ]:
# ── 7. Section coverage + chunk-size distribution ─────────────────────────
divider("SECTION COVERAGE SUMMARY")

seen: Dict[str, Dict] = {}
for c in chunks:
    name = c.section_name or "(fixed-size fallback)"
    if name not in seen:
        seen[name] = {"chunks": 0, "chars": 0}
    seen[name]["chunks"] += 1
    seen[name]["chars"]  += len(c.text)

print(f"  {'Section':<45s} {'Chunks':>7s} {'Chars':>10s}")
print(f"  {'-'*45} {'-'*7} {'-'*10}")
for name, info in sorted(seen.items(), key=lambda x: -x[1]["chunks"]):
    print(f"  {name:<45s} {info['chunks']:>7,d} {info['chars']:>10,d}")

lengths = [len(c.text) for c in chunks]
print(f"\n  Total  chunks : {len(chunks):,}")
print(f"  Total  chars  : {sum(lengths):,}")
print(f"  Avg chunk     : {sum(lengths)/len(lengths):,.0f} chars")
print(f"  Min / Max     : {min(lengths):,} / {max(lengths):,} chars")

In [ ]:
# ── 8. Overlap verification ────────────────────────────────────────────────
divider("OVERLAP VERIFICATION")

gaps, checked = 0, 0
for i in range(1, len(chunks)):
    prev, cur = chunks[i-1], chunks[i]
    # Only verify consecutive chunks in the same document + section
    if (prev.source_document_id != cur.source_document_id or
            prev.section_name != cur.section_name):
        continue
    checked += 1
    overlap_chars = prev.end_char - cur.start_char
    if overlap_chars < 0:
        gaps += 1
        if gaps <= 5:
            print(f"  ⚠ GAP  chunks {i-1}→{i} ({prev.section_name or 'fixed-size'}): "
                  f"{-overlap_chars} chars")

print(f"\n  Pairs checked : {checked:,}")
if gaps == 0:
    print("  ✅  All same-section adjacent chunks overlap correctly.")
else:
    print(f"  ❌  {gaps} pair(s) have gaps — review boundary logic.")

In [ ]:
# ── 9. Generate embeddings ─────────────────────────────────────────────────
divider("STEP 4 — EMBEDDING GENERATION")

embedder = EmbeddingGenerator(
    model_name=EMBED_MODEL,
    normalize_embeddings=True,
)

embedded_chunks: List[EmbeddedChunk] = embedder.run(chunks, batch_size=EMBED_BATCH)

ok = embedder.verify()
print(f"\n  Chunks embedded  : {len(embedded_chunks):,}")
print(f"  Embedding dim    : {embedder.embedding_dim}")
print(f"  Verify passed    : {ok}")
print("\n  Statistics:")
for k, v in embedder.statistics().items():
    print(f"    {k}: {v}")

## Vector Store — Redis Stack with SQLite fallback

The next cell probes Redis Stack. Depending on what it finds:

| Condition | Backend used |
|---|---|
| Redis Stack reachable + RediSearch loaded | **Redis** — HNSW / COSINE index |
| Redis unreachable, or RediSearch module missing | **SQLite** — embeddings stored as BLOB columns; KNN via NumPy cosine similarity |

The `BACKEND` variable is set to `"redis"` or `"sqlite"` and used by all subsequent cells.

In [ ]:
# ── 10. Vector store connection — Redis with SQLite fallback ──────────────
import redis as _redis_mod

BACKEND: str     = "sqlite"    # resolved below
r:       object  = None        # redis.Redis client (set only when BACKEND=="redis")
VECTOR_DIM: int  = embedder.embedding_dim


# ── helper: pack float list → FLOAT32 bytes (used by both backends) ────────
def vec_to_bytes(vec: List[float]) -> bytes:
    return struct.pack(f"{len(vec)}f", *vec)

def bytes_to_vec(b: bytes) -> np.ndarray:
    n = len(b) // 4
    return np.array(struct.unpack(f"{n}f", b), dtype="float32")


# ── 10a. Try Redis Stack ───────────────────────────────────────────────────
def _try_redis() -> bool:
    """Return True and configure global `r` if Redis Stack is usable."""
    global r, BACKEND
    try:
        client = _redis_mod.Redis(
            host=REDIS_HOST, port=REDIS_PORT, db=REDIS_DB,
            decode_responses=False,
            socket_connect_timeout=2,
        )
        client.ping()
    except (_redis_mod.exceptions.ConnectionError,
            _redis_mod.exceptions.TimeoutError) as exc:
        print(f"  [info] Redis unreachable ({exc}) — will use SQLite fallback.")
        return False

    # Verify RediSearch module is loaded
    modules = {m[b"name"].decode() if isinstance(m[b"name"], bytes) else m[b"name"]
               for m in client.module_list()}
    if "search" not in modules and "bf" not in modules and "ReJSON" not in modules:
        # module_list names vary; check FT.INFO as a reliable probe
        try:
            client.execute_command("FT._LIST")
        except _redis_mod.exceptions.ResponseError as exc:
            print(f"  [info] RediSearch not available ({exc}) — will use SQLite fallback.")
            return False

    r       = client
    BACKEND = "redis"
    return True


# ── 10b. SQLite embedding store schema ────────────────────────────────────
def _init_sqlite_embedding_store(db_path: str) -> sqlite3.Connection:
    """Add an `embedded_chunks` table to the existing SQLite DB."""
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS embedded_chunks (
            chunk_id           TEXT PRIMARY KEY,
            source_document_id TEXT NOT NULL,
            section_name       TEXT,
            text               TEXT,
            embedding_model    TEXT,
            year               TEXT,
            cik                TEXT,
            naics3             TEXT,
            sic                TEXT,
            chunk_index        INTEGER,
            total_chunks       INTEGER,
            start_char         INTEGER,
            end_char           INTEGER,
            embedding          BLOB NOT NULL,
            created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_ec_doc ON embedded_chunks(source_document_id)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_ec_section ON embedded_chunks(section_name)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_ec_year ON embedded_chunks(year)")
    conn.commit()
    return conn


# ── Resolve backend ────────────────────────────────────────────────────────
redis_ok = _try_redis()

if BACKEND == "redis":
    from redis.commands.search.field           import TextField, TagField, NumericField, VectorField
    from redis.commands.search.indexDefinition import IndexDefinition, IndexType
    from redis.commands.search.query           import Query

    try:
        r.ft(REDIS_INDEX).dropindex(delete_documents=False)
        print(f"  Dropped existing Redis index '{REDIS_INDEX}'")
    except Exception:
        pass

    schema = [
        TextField("chunk_id"), TextField("source_document_id"),
        TextField("section_name"), TextField("text"),
        TagField("embedding_model"), TagField("year"),
        TagField("cik"), TagField("naics3"), TagField("sic"),
        NumericField("chunk_index"), NumericField("total_chunks"),
        NumericField("start_char"),  NumericField("end_char"),
        VectorField(
            "embedding", "HNSW",
            {"TYPE": "FLOAT32", "DIM": VECTOR_DIM,
             "DISTANCE_METRIC": "COSINE", "M": 16, "EF_CONSTRUCTION": 200},
        ),
    ]
    r.ft(REDIS_INDEX).create_index(
        schema,
        definition=IndexDefinition(prefix=[REDIS_PREFIX], index_type=IndexType.HASH),
    )
    print(f"  ✅  Redis backend ready — index '{REDIS_INDEX}'  (HNSW, COSINE, dim={VECTOR_DIM})")

else:
    _sqlite_embed_conn = _init_sqlite_embedding_store(SQLITE_PATH)
    print(f"  ✅  SQLite fallback ready — table 'embedded_chunks' in {SQLITE_PATH}")

print(f"\n  BACKEND = {BACKEND!r}")

In [ ]:
# ── 11. Store embedded chunks ─────────────────────────────────────────────
divider(f"STEP 5 — STORING EMBEDDINGS  (backend={BACKEND!r})")


def _store_redis(embedded: List[EmbeddedChunk]) -> int:
    """Write chunks to Redis via pipeline."""
    pipe    = r.pipeline(transaction=False)
    stored  = 0
    PIPE_SZ = 500
    for ec in embedded:
        meta = ec.metadata or {}
        pipe.hset(
            f"{REDIS_PREFIX}{ec.chunk_id}",
            mapping={
                "chunk_id":           ec.chunk_id,
                "source_document_id": ec.source_document_id,
                "section_name":       ec.section_name or "",
                "text":               ec.text,
                "embedding_model":    ec.embedding_model,
                "year":               str(meta.get("year",   "")),
                "cik":                str(meta.get("cik",    "")),
                "naics3":             str(meta.get("naics3", "")),
                "sic":                str(meta.get("sic",    "")),
                "chunk_index":        int(meta.get("chunk_index",  0)),
                "total_chunks":       int(meta.get("total_chunks", 1)),
                "start_char":         int(meta.get("start_char",   0)),
                "end_char":           int(meta.get("end_char",     0)),
                "embedding":          vec_to_bytes(ec.embedding),
            },
        )
        stored += 1
        if stored % PIPE_SZ == 0:
            pipe.execute()
            pipe = r.pipeline(transaction=False)
            print(f"  {stored:,}/{len(embedded):,} …")
    pipe.execute()
    return stored


def _store_sqlite(embedded: List[EmbeddedChunk], conn: sqlite3.Connection) -> int:
    """Upsert chunks into SQLite embedded_chunks table."""
    rows = []
    for ec in embedded:
        meta = ec.metadata or {}
        rows.append((
            ec.chunk_id,
            ec.source_document_id,
            ec.section_name or "",
            ec.text,
            ec.embedding_model,
            str(meta.get("year",   "")),
            str(meta.get("cik",    "")),
            str(meta.get("naics3", "")),
            str(meta.get("sic",    "")),
            int(meta.get("chunk_index",  0)),
            int(meta.get("total_chunks", 1)),
            int(meta.get("start_char",   0)),
            int(meta.get("end_char",     0)),
            vec_to_bytes(ec.embedding),
        ))
    conn.executemany("""
        INSERT OR REPLACE INTO embedded_chunks
        (chunk_id, source_document_id, section_name, text, embedding_model,
         year, cik, naics3, sic,
         chunk_index, total_chunks, start_char, end_char, embedding)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, rows)
    conn.commit()
    return len(rows)


if BACKEND == "redis":
    stored = _store_redis(embedded_chunks)
else:
    stored = _store_sqlite(embedded_chunks, _sqlite_embed_conn)

print(f"\n  ✅  Stored {stored:,} chunks  →  {BACKEND}")

In [ ]:
# ── 12. Store info ────────────────────────────────────────────────────────
def _str(v):
    return v.decode() if isinstance(v, bytes) else str(v)

if BACKEND == "redis":
    info = r.ft(REDIS_INDEX).info()
    print(f"  Backend            : Redis Stack")
    print(f"  Index name         : {_str(info.get('index_name', ''))}")
    print(f"  Num docs indexed   : {info.get('num_docs', 0)}")
    print(f"  Hash indexing err  : {info.get('hash_indexing_failures', 0)}")
    print(f"  Keys in Redis      : {len(r.keys(REDIS_PREFIX + '*')):,}")
else:
    n = _sqlite_embed_conn.execute("SELECT COUNT(*) FROM embedded_chunks").fetchone()[0]
    dims = _sqlite_embed_conn.execute(
        "SELECT LENGTH(embedding)/4 FROM embedded_chunks LIMIT 1"
    ).fetchone()
    print(f"  Backend            : SQLite fallback  ({SQLITE_PATH})")
    print(f"  Rows in table      : {n:,}")
    print(f"  Embedding dim      : {dims[0] if dims else '?'}")

In [ ]:
# ── 13. KNN vector search ─────────────────────────────────────────────────
divider(f"STEP 6 — KNN VECTOR SEARCH  (backend={BACKEND!r})")

QUERY_TEXT = "liquidity and capital resources cash flow"
TOP_K      = 5

query_vec   = embedder.encode_one(QUERY_TEXT)
query_bytes = vec_to_bytes(query_vec)
query_arr   = np.array(query_vec, dtype="float32")


def _knn_redis(query_b: bytes, k: int, extra_filter: str = "*") -> List[Dict]:
    q = (
        Query(f"{extra_filter}=>[KNN {k} @embedding $vec AS score]")
        .sort_by("score")
        .return_fields("score", "chunk_id", "section_name", "text", "year", "cik")
        .dialect(2)
    )
    results = r.ft(REDIS_INDEX).search(q, query_params={"vec": query_b})
    return [
        {
            "score":        getattr(d, "score",        "N/A"),
            "chunk_id":     getattr(d, "chunk_id",     ""),
            "section_name": getattr(d, "section_name", ""),
            "text":         getattr(d, "text",         ""),
            "year":         getattr(d, "year",         ""),
            "cik":          getattr(d, "cik",          ""),
        }
        for d in results.docs
    ]


def _knn_sqlite(
    q_arr: np.ndarray, k: int,
    section_filter: Optional[str] = None,
    year_filter:    Optional[str] = None,
) -> List[Dict]:
    """Brute-force cosine KNN over SQLite BLOB embeddings via NumPy."""
    sql  = "SELECT chunk_id, source_document_id, section_name, text, year, cik, embedding FROM embedded_chunks"
    cond, params = [], []
    if section_filter:
        cond.append("section_name = ?"); params.append(section_filter)
    if year_filter:
        cond.append("year = ?");         params.append(year_filter)
    if cond:
        sql += " WHERE " + " AND ".join(cond)

    rows = _sqlite_embed_conn.execute(sql, params).fetchall()
    if not rows:
        return []

    # Stack all embeddings, compute cosine similarity in one shot
    ids   = [(row[0], row[1], row[2], row[3], row[4], row[5]) for row in rows]
    mat   = np.stack([bytes_to_vec(row[6]) for row in rows])    # (N, dim)
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    q_n   = q_arr / (np.linalg.norm(q_arr) + 1e-12)
    sims  = (mat / norms) @ q_n                                  # cosine similarity

    top_idx = np.argsort(sims)[::-1][:k]
    return [
        {
            "score":        f"{1 - sims[i]:.6f}",   # distance (lower = better)
            "chunk_id":     ids[i][0],
            "section_name": ids[i][2],
            "text":         ids[i][3],
            "year":         ids[i][4],
            "cik":          ids[i][5],
        }
        for i in top_idx
    ]


# Run search
if BACKEND == "redis":
    hits = _knn_redis(query_bytes, TOP_K)
else:
    hits = _knn_sqlite(query_arr, TOP_K)

print(f"  Query : {QUERY_TEXT!r}")
print(f"  Top-{TOP_K} results:\n")
for rank, h in enumerate(hits, 1):
    section = h['section_name'] or '(fixed-size fallback)'
    snippet = h['text'][:160].replace('\n', ' ') + ('…' if len(h['text']) > 160 else '')
    print(f"  #{rank}  score={h['score']}  [{section}]  cik={h['cik']}  year={h['year']}")
    print(f"       {snippet}\n")

In [ ]:
# ── 14. Filtered KNN search (section + year) ───────────────────────────────
divider("FILTERED KNN SEARCH")

FILTER_SECTION = "Results Of Operations"
FILTER_YEAR    = "2024"
FILTER_K       = 3

try:
    if BACKEND == "redis":
        section_tag = FILTER_SECTION.replace(" ", "\\ ")
        filt        = f"(@section_name:{{{section_tag}}}) @year:{{{FILTER_YEAR}}}"
        hits        = _knn_redis(query_bytes, FILTER_K, extra_filter=filt)
    else:
        hits = _knn_sqlite(
            query_arr, FILTER_K,
            section_filter=FILTER_SECTION,
            year_filter=FILTER_YEAR,
        )

    print(f"  Filter: section='{FILTER_SECTION}'  year='{FILTER_YEAR}'")
    print(f"  Top-{FILTER_K} results:\n")
    if not hits:
        print("  (no results matched the filter)")
    for rank, h in enumerate(hits, 1):
        section = h['section_name'] or '(fixed-size fallback)'
        snippet = h['text'][:160].replace('\n', ' ') + ('…' if len(h['text']) > 160 else '')
        print(f"  #{rank}  score={h['score']}  [{section}]")
        print(f"       {snippet}\n")

except Exception as exc:
    print(f"  [warn] Filtered search error: {exc}")

In [ ]:
# ── 15. JSON export (offline backup / inspection) ──────────────────────────
export = [
    {
        "chunk_id":           ec.chunk_id,
        "source_document_id": ec.source_document_id,
        "section_name":       ec.section_name,
        "text":               ec.text,
        "embedding_model":    ec.embedding_model,
        "embedding":          ec.embedding,
        "metadata":           ec.metadata,
    }
    for ec in embedded_chunks
]

with open(OUTPUT_JSON, "w", encoding="utf-8") as fh:
    json.dump(export, fh, ensure_ascii=False, default=str)

size_mb = Path(OUTPUT_JSON).stat().st_size / 1_048_576
print(f"Exported {len(export):,} embedded chunks → {OUTPUT_JSON}  ({size_mb:.1f} MB)")

In [ ]:
# ── 16. Pipeline summary ───────────────────────────────────────────────────
divider("PIPELINE SUMMARY")
fallback_pct = 100 * len(_fallback_docs) / max(len(documents), 1)
print(f"  Source documents    : {len(documents):,}")
print(f"  Tables extracted    : {len(all_tables):,}   → {SQLITE_PATH} (tables_long / filings_tables)")
print(f"  Text chunks         : {len(chunks):,}")
print(f"    MDA section-aware : {sum(1 for c in chunks if c.section_name):,}")
print(f"    Fixed-size fallbk : {sum(1 for c in chunks if not c.section_name):,}  "
      f"({fallback_pct:.1f}% of docs used fallback)")
print(f"  Embedded chunks     : {len(embedded_chunks):,}  (dim={VECTOR_DIM})")
print(f"  Embed model         : {EMBED_MODEL}")
print()
if BACKEND == "redis":
    print(f"  Vector store        : Redis Stack  {REDIS_HOST}:{REDIS_PORT}")
    print(f"  Index               : {REDIS_INDEX}  (HNSW, COSINE)")
else:
    print(f"  Vector store        : SQLite fallback  ({SQLITE_PATH})")
    print(f"  Table               : embedded_chunks  (KNN via NumPy cosine)")
print(f"  JSON backup         : {OUTPUT_JSON}")
print("\n  Done. ✅")